# 01 · Data audit & cleaning
**Report section 2 — Data Cleaning & Feature Engineering.**

Goals: understand the raw schema, quantify missingness/duplicates/inconsistencies, justify every cleaning decision.
Reusable logic lives in `superstore.data`; this notebook documents and *justifies* it.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from superstore import config
from superstore.viz.plots import set_style

set_style()
pd.set_option("display.max_columns", 50)

In [2]:
from superstore.data.clean import audit, clean_orders
from superstore.data.load import load_raw

raw = load_raw()
print(raw.shape)
raw.head()

(51290, 24)


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,city,state,country,postal_code,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.184509,933.570007,Critical
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,<NA>,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.765015,923.630005,Critical
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,<NA>,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.971008,915.489990,Medium
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,<NA>,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.540001,910.159973,Medium
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,<NA>,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.519989,903.039978,Critical


In [3]:
audit(raw)

,dtype,n_missing,pct_missing,n_unique,example,pct_unique
row_id,int32,0,0.00,51290,32298,100.00
order_id,str,0,0.00,25035,CA-2012-124891,48.81
order_date,datetime64[us],0,0.00,1430,2012-07-31 00:00:00,2.79
ship_date,datetime64[us],0,0.00,1464,2012-07-31 00:00:00,2.85
ship_mode,category,0,0.00,4,Same Day,0.01
customer_id,category,0,0.00,1590,RH-19495,3.10
customer_name,category,0,0.00,795,Rick Hansen,1.55
segment,category,0,0.00,3,Consumer,0.01
city,str,0,0.00,3636,New York City,7.09
state,category,0,0.00,1094,New York,2.13


## Checks to document
- [x] Duplicated order lines? Duplicated `order_id` + `product_id`? No exact duplicate rows. 38 repeated `order_id` + `product_id` pairs (76 rows) are not exact copies (they differ in quantity or shipping cost), so they are kept and flagged with `flag_dup_key`.
- [x] Date range, ship_date ≥ order_date, ship-mode vs ship-days consistency: orders span 2011-01-01 to 2014-12-31; no ship date precedes its order date; `ship_days` is consistent with `ship_mode` (Same Day 0–1, First Class 1–3, Second Class 2–5, Standard Class 4–7 days).
- [x] Missing values (e.g. `postal_code` outside the US) — MCAR/MAR/MNAR? strategy? only `postal_code` (80.5%), missing for every non-US row and present for every US row, so it is structurally missing (determined by `country`) and is not imputed.
- [x] Negative profit / extreme sales: errors or real? (flag vs. cap vs. drop): 24.5% of lines are losses, almost all on discounted items; extremes (max sales 22,638, min profit −6,600) are large real orders, so they are flagged, not capped or dropped.
- [x] Discount distribution: discrete levels? per market? discrete levels (23 distinct values after rounding to 2 decimals) that differ by market (Canada none, Africa 3 levels, APAC 19); a float32 bin-edge bug was fixed by converting to float64 and rounding before binning.
- [x] Memory before/after `optimise_dtypes`: 17.9 MB → 8.2 MB (54% saved).

In [4]:
orders = clean_orders(raw)

In [5]:
# Cleaning log and memory:
print(orders.attrs["cleaning_log"])
raw_mb, clean_mb = raw.memory_usage(deep=True).sum()/1e6, orders.memory_usage(deep=True).sum()/1e6
print(f"raw {raw_mb:.1f} MB → clean {clean_mb:.1f} MB ({100*(1-clean_mb/raw_mb):.0f}% saved)")

{'rows_raw': 51290, 'dropped_exact_dups': 0, 'dropped_missing_key': 0, 'rows_clean': 51290}
raw 8.2 MB → clean 14.8 MB (-80% saved)


In [6]:
# Flags and coverage:
print(orders[[c for c in orders.columns if c.startswith("flag_")]].mean())
for col in ("sales", "profit"):
    f = orders[f"flag_outlier_{col}"]
    print(f"{col}: {f.mean():.1%} of rows, {orders.loc[f, col].abs().sum()/orders[col].abs().sum():.1%} of total |{col}|")

flag_dup_key               0.001482
flag_order_id_collision    0.052700
flag_ship_before_order     0.000000
flag_outlier_sales         0.110255
flag_outlier_profit        0.190193
dtype: float64
sales: 11.0% of rows, 56.1% of total |sales|
profit: 19.0% of rows, 74.6% of total |profit|


Interpretation of the previous results:
- 0.1482% of repeated order-product combinations
- 0% of shipping dates before order dates
- 11.03% of sales values outside the IQR bound
- 19.02% of profit values ouside the IQR bounds

In [7]:
# Repeated lines (evidence they are not exact copies):
d = orders[orders.flag_dup_key]
g = d.groupby(["order_id", "product_id"], observed=True)
print(g.ngroups, "pairs", len(d), "rows")                                                   # 38 pairs, 76 rows
cols = ["sales", "quantity", "profit", "shipping_cost"]
print("pairs identical on all of", cols, ":", g[cols].nunique().eq(1).all(axis=1).sum())    # 0 → no exact copies

38 pairs 76 rows
pairs identical on all of ['sales', 'quantity', 'profit', 'shipping_cost'] : 0


In [8]:
# Hidden blanks and validation:
txt = orders.select_dtypes(include=["object", "string", "category"]).columns
print({c: int(orders[c].astype("string").str.strip().eq("").sum()) for c in txt} )

{'order_id': 0, 'ship_mode': 0, 'customer_id': 0, 'customer_name': 0, 'segment': 0, 'city': 0, 'state': 0, 'country': 0, 'postal_code': 0, 'market': 0, 'region': 0, 'product_id': 0, 'category': 0, 'sub_category': 0, 'product_name': 0, 'order_priority': 0, 'order_key': 0, 'product_key': 0, 'discount_band': 0}


In [9]:
# Discount levels per market
print(orders.assign(has_disc=orders.discount > 0)
      .groupby("market", observed=True)
      .agg(rows=("discount", "size"), pct_discounted=("has_disc", "mean"),
           levels=("discount", "nunique"), max_disc=("discount", "max")).round(3))

         rows  pct_discounted  levels  max_disc
market                                         
APAC    11002           0.574      19      0.80
Africa   4587           0.230       3      0.70
Canada    384           0.000       1      0.00
EMEA     5029           0.320       4      0.70
EU      10000           0.387      14      0.85
LATAM   10294           0.367       7      0.80
US       9994           0.520      12      0.80


In [11]:
# Memory before/after:
raw_plain = load_raw(optimise=False)                         # same loading and date parsing, no dtype optimisation
common = [c for c in orders.columns if c in raw_plain.columns]
before = raw_plain[common].memory_usage(deep=True).sum() / 1e6
after  = orders[common].memory_usage(deep=True).sum() / 1e6
print(f"before {before:.1f} MB → after {after:.1f} MB ({100*(1-after/before):.0f}% saved) in this environment.")

orders.to_parquet(config.ORDERS_FILE, index=False)

before 17.9 MB → after 8.2 MB (54% saved) in this environment.
